# MANIA v1.2 — 2026-06-26
# MANIA_preprocessing

**Input:** trajectory files (GROMACS `.tpr`/`.xtc`, NAMD `.psf`/`.dcd`)
**Output:** `mania_exports/`

## Pipeline
```
Step 1.  Package installation
Step 2.  Imports
Step 3.  Configuration & paths
Step 3b. Residue Library QC            ← NEW in v1.1
Step 4.  Trajectory loading (MDAnalysis Universe)
Step 5.  ESM-2 embeddings (optional, USE_ESM2=False)
Step 6.  Structural features (RMSF, SASA, DSSP, Φ/Ψ, TM-Z, Rg(t))  ← Rg NEW
Step 7.  Protein–protein contacts (per-condition)
Step 7b. Per-frame contact export (contacts_perframe parquet)
Step 8.  Cα alignment (Kabsch) & node coordinates
Step 9.  Non-protein nodes (lipids, glycans, ligands)
Step 10. HeteroGraph (optional, USE_HETERO=True)
Step 11. Artefact export (residue_table, contact_edges, rg_timeseries, manifest) ← Rg NEW
Step 12. Upload to Yandex.Disk
```

> **Изменения v1.1 vs v1.0:**
> - **Step 3b** — автоматическая валидация residue library (`mania_residue_library.json`) против Universe до старта расчётов; ошибки блокируют выполнение
> - **Step 6** — добавлен `compute_rg_timeseries()`: per-frame Rg(t), mean/std по белку
> - **Step 10** — `data.global_features` = [rg_mean, rg_std] при USE_HETERO=True
> - **Step 11** — новый артефакт `rg_timeseries_{condition}.csv`; `mania_manifest.json` содержит секцию `global_features`
> - **Step 12** — rg_timeseries CSV включён в загрузку

> **Изменения v1.2 vs v1.1:**
> - **Step 7**   Полный рефакторинг алгоритма контактов
              • Класс InteractionAccumulator заменён на dataclass EdgeAccumulator
              • Введён buildatomcache() — атомные контексты вычисляются один раз
                до цикла по фреймам, а не при каждой итерации
              • Backbone вынесен в отдельный шаг Step 7 (не входит в Step 7b)
              • Детектирование H-bonds теперь учитывает атомы водорода (H-atom)
                из bonds-топологии вместо геометрического приближения
              • Ароматическое π–π взаимодействие: добавлена проверка угла нормали
                (parallel / T-shape) через SVD кольца
              • Cation–π: добавлена проверка угла catpos → ring-normal
              • Ionic / salt bridge разделены через единую frameioniccache()
                с параметром cutoffkey
> - **Step 7b**  Пофреймовый экспорт теперь включает backbone; подсчёт
              детализирован по типам (backbone / hbond / vdw / hydrophobic)

> **Ограничения по размеру системы (рекомендуемые):**
> - Максимум кадров: 10 000 (при stride=1). При большем числе используйте `frame_stride_contacts ≥ 3`.
> - Максимум остатков: 2 000 белковых остатков.
> - Для steered MD с лигандом: укажите имена остатков лиганда в `LIGAND_RESNAMES`.


## Step 1. Package installation

In [ ]:
import os, sys, subprocess
_USE_HETERO_ENV = os.environ.get("USE_HETERO", "false").lower() == "true"

PACKAGES = [
    "mdanalysis", "biopython", "freesasa",
    "pandas", "numpy", "scipy", "networkx", "tqdm",
    "matplotlib", "seaborn", "pyarrow",
]
OPTIONAL = ["fair-esm", "yadisk"]
HETERO_PACKAGES = ["torch", "torch-geometric"]

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    for pkg in PACKAGES + OPTIONAL:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
    if _USE_HETERO_ENV:
        for pkg in HETERO_PACKAGES:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
        print("Colab: все пакеты (включая PyG) установлены.")
    else:
        print("Colab: базовые пакеты установлены. PyTorch Geometric пропущен (USE_HETERO=False).")
else:
    print("Локальный запуск: предполагается, что зависимости установлены.")
    print("Рекомендуем: conda env create -f environment.yml")


Colab: базовые пакеты установлены. PyTorch Geometric пропущен (USE_HETERO=False).


## Step 2. Imports

In [ ]:
import os, gc, json, math, shutil, warnings
from pathlib import Path
from dataclasses import dataclass, field
from collections import defaultdict
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import pyarrow as pa
import pyarrow.parquet as pq

import MDAnalysis as mda
from MDAnalysis.analysis import rms, dihedrals
from MDAnalysis.lib.distances import capped_distance, distance_array, calc_bonds, calc_angles

try:
    from google.colab import userdata as _colab_userdata
    IN_COLAB = True
except ImportError:
    _colab_userdata = None
    IN_COLAB = False

warnings.filterwarnings('ignore')
np.set_printoptions(suppress=True, precision=4)
pd.set_option('display.max_columns', 200)
print("Imports OK.")
try:
    import torch
    from torch import Tensor
    from torch_geometric.data import HeteroData
    _TORCH_OK = True
    print("PyTorch / PyG: OK")
except ImportError:
    _TORCH_OK = False
    print("PyTorch / PyG не установлены. USE_HETERO должен быть False.")


Imports OK.
PyTorch / PyG не установлены. USE_HETERO должен быть False.


## Step 3. Configuration & paths

In [ ]:
# ══════════════════════════════════════════════════════════════════
# КОНФИГУРАЦИЯ — редактируй этот блок перед запуском
# ══════════════════════════════════════════════════════════════════

# ── Источник данных: 'local' | 'yadisk' | 'gdrive' ─────────────
SOURCE = 'yadisk'

# ── Пути к траекториям (при SOURCE='local') ─────────────────────
LOCAL_CONFIG = {
    'normal': {
        'topology':   'data/normal/step7_production.tpr',
        'trajectory': 'data/normal/trajectory.xtc',
        'label': 0,
        'ph': 7.4,
    },
    'tumor': {
        'topology':   'data/tumor/step7_production.tpr',
        'trajectory': 'data/tumor/trajectory.xtc',
        'label': 1,
        'ph': 6.8,
    },
}

# ── Яндекс Диск (при SOURCE='yadisk') ───────────────────────────
MW_TOKEN = _colab_userdata.get('MW_TOKEN') if IN_COLAB and _colab_userdata else None
YADISK_BASE = 'MANIA_WANIA_project/MD_trajectories/NaPi2b/Ramilia/30ns_03-2026'
YADISK_PATHS = {
    'normal': {
        'topology':   f'{YADISK_BASE}/NORM/03_PRODUCTION/step7_production_10ns.tpr',
        'trajectory': f'{YADISK_BASE}/NORM/03_PRODUCTION/step_full_0_30ns.xtc',
        'label': 0, 'ph': 7.4,
    },
    'tumor': {
        'topology':   f'{YADISK_BASE}/TUMOR/03_PRODUCTION/step7_production_10ns.tpr',
        'trajectory': f'{YADISK_BASE}/TUMOR/03_PRODUCTION/step_full_0_30ns.xtc',
        'label': 1, 'ph': 6.8,
    },
}

# ── Биохимия системы ────────────────────────────────────────────
PROTEIN_SELECTION  = 'protein'
CA_SELECTION       = 'protein and name CA'
LIPID_SELECTION    = 'resname POPC POPE POPS PSM POPI CHL1 GLPA CER160'
GLYCAN_RESNAMES    = {'BGLC', 'BGLCNA', 'BMAN', 'BGAL', 'BGALNA', 'AFUC', 'ANE5AC'}
LIGAND_RESNAMES    = set()
FA2G2S2_TEMPLATE   = ['BGLCNA', 'BGLCNA', 'BMAN', 'BMAN', 'BMAN', 'BMAN',
                       'BGALNA', 'BGALNA', 'AFUC', 'ANE5AC', 'ANE5AC']
DEFAULT_GLYCAN_ANCHORS = [295, 308]

# ── Регионы интереса ─────────────────────────────────────────────
REGIONS_OF_INTEREST = {
    'epitope':      (311, 341),
    'ECD':          (234, 361),
    'full_protein': None,
}
BACKBONE_MAX_CA_DIST_A = 4.5

# ── Пути вывода ──────────────────────────────────────────────────
OUTPUT_DIR = Path('/content/mania_outputs') if IN_COLAB else Path.cwd() / 'mania_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR = OUTPUT_DIR / 'mania_exports'
EXPORT_DIR.mkdir(exist_ok=True)

# ── Residue library QC (v1.1) ───────────────────────────────────
YADISK_LIB_PATH  = "/MANIA_WANIA_project/auxiliary_files/residue_library/mania_residue_library.json"
RESIDUE_LIB_PATH = EXPORT_DIR / "mania_residue_library.json"
RESIDUE_LIB_VALIDATE  = True   # False — пропустить QC полностью
RESIDUE_LIB_FAIL_ON_ERROR = True  # True — RuntimeError при статусе 'fail' или 'not_found'

# ── Rg (v1.1) ───────────────────────────────────────────────────
COMPUTE_RG = True   # False — пропустить расчёт Rg(t)

# ── Параметры расчёта взаимодействий ─────────────────────────────
# Быстрый прогон (отладка): frame_stride_contacts=10, sasa_frame_stride=100
# Финальный прогон: frame_stride_contacts=1, sasa_frame_stride=50
FEATURE_CALC_PARAMS = {
    'frame_stride_contacts':   10,
    'frame_stride_features':   10,
    'sasa_frame_stride':       50,
    'frame_frac':              0.10,
    'min_contact_frequency':   0.30,
    'hbond_distance_cutoff_A': 3.5,
    'hbond_angle_cutoff_deg':  120.0,
    'disulfide_sg_cutoff_A':   2.2,
    'vdw_distance_min_A':      3.0,
    'vdw_distance_max_A':      4.5,
    'hydrophobic_cutoff_A':    5.0,
    'aromatic_pi_centroid_cutoff_A':    7.0,
    'aromatic_parallel_angle_max_deg':  30.0,
    'aromatic_tshape_angle_min_deg':    60.0,
    'aromatic_tshape_angle_max_deg':    120.0,
    'cation_pi_cutoff_A':               6.0,
    'cation_pi_normal_angle_max_deg':   60.0,
    'ionic_cutoff_A':          6.0,
    'salt_bridge_cutoff_A':    4.0,
    'protein_lipid_cutoff_A':  6.0,
    'protein_glycan_cutoff_A': 6.0,
}

print(f"SOURCE      = {SOURCE}")
print(f"OUTPUT_DIR  = {OUTPUT_DIR}")
print(f"EXPORT_DIR  = {EXPORT_DIR}")
print(f"FEATURE_CALC_PARAMS: {len(FEATURE_CALC_PARAMS)} параметров")
print(f"COMPUTE_RG  = {COMPUTE_RG}")
print(f"RESIDUE_LIB_VALIDATE = {RESIDUE_LIB_VALIDATE}")

SOURCE      = yadisk
OUTPUT_DIR  = /content/mania_outputs
EXPORT_DIR  = /content/mania_outputs/mania_exports
FEATURE_CALC_PARAMS: 21 параметров
COMPUTE_RG  = True
RESIDUE_LIB_VALIDATE = True


In [ ]:
AA3_TO_1 = {
    'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C',
    'GLN':'Q','GLU':'E','GLY':'G','HIS':'H','HID':'H',
    'HIE':'H','HIP':'H','HSD':'H','HSE':'H','HSP':'H',
    'ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F',
    'PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V',
}

EDGE_SEMANTICS = {
    'backbone':     {'criterion': 'sequential i→i+1 Cα',
                     'distance_A': BACKBONE_MAX_CA_DIST_A},
    'hbond':        {'criterion': 'distance + angle',
                     'distance_A': FEATURE_CALC_PARAMS['hbond_distance_cutoff_A'],
                     'angle_deg_min': FEATURE_CALC_PARAMS['hbond_angle_cutoff_deg']},
    'disulfide':    {'criterion': 'SG-SG distance',
                     'distance_A': FEATURE_CALC_PARAMS['disulfide_sg_cutoff_A']},
    'vdw':          {'criterion': 'heavy-atom distance in window',
                     'distance_min_A': FEATURE_CALC_PARAMS['vdw_distance_min_A'],
                     'distance_max_A': FEATURE_CALC_PARAMS['vdw_distance_max_A']},
    'hydrophobic':  {'criterion': 'CB-CB distance, hydrophobic residue pair',
                     'distance_A': FEATURE_CALC_PARAMS['hydrophobic_cutoff_A']},
    'aromatic_pi':  {'criterion': 'ring centroid distance, aromatic pair',
                     'distance_A': FEATURE_CALC_PARAMS['aromatic_pi_centroid_cutoff_A']},
    'cation_pi':    {'criterion': 'cation to aromatic ring centroid',
                     'distance_A': FEATURE_CALC_PARAMS['cation_pi_cutoff_A']},
    'ionic':        {'criterion': 'charged side-chain heavy atom distance',
                     'distance_A': FEATURE_CALC_PARAMS['ionic_cutoff_A']},
    'salt_bridge':  {'criterion': 'donor/acceptor charged atom distance',
                     'distance_A': FEATURE_CALC_PARAMS['salt_bridge_cutoff_A']},
    'protein_lipid':  {'criterion': 'min protein to lipid heavy atom',
                       'distance_A': FEATURE_CALC_PARAMS['protein_lipid_cutoff_A']},
    'glycan_anchor':  {'criterion': 'covalent N-glycosylation anchor edge',
                       'distance_A': 0.0},
    'protein_glycan': {'criterion': 'min protein to glycan heavy atom',
                       'distance_A': FEATURE_CALC_PARAMS['protein_glycan_cutoff_A']},
    'protein_ligand': {'criterion': 'min protein to ligand heavy atom',
                       'distance_A': FEATURE_CALC_PARAMS['protein_glycan_cutoff_A']},
}

with open(EXPORT_DIR / 'edge_semantics.json', 'w') as f:
    json.dump(EDGE_SEMANTICS, f, indent=2, ensure_ascii=False)

print("EDGE_SEMANTICS:", list(EDGE_SEMANTICS.keys()))


EDGE_SEMANTICS: ['backbone', 'hbond', 'disulfide', 'vdw', 'hydrophobic', 'aromatic_pi', 'cation_pi', 'ionic', 'salt_bridge', 'protein_lipid', 'glycan_anchor', 'protein_glycan', 'protein_ligand']


In [ ]:
# ── Step 3b: Residue Library QC ──────────────────────────────────────────────

def run_residue_library_qc(
    universe_dict: dict,
    library_json: Path,
    export_dir: Path,
    fail_on_error: bool = True,
) -> dict:
    """
    Validates all residue names found in universes against the MANIA residue library.

    Returns dict: {resname: {'status': str, 'coverage': float, 'block_type': str}}
    Raises RuntimeError if fail_on_error=True and any residue has status 'fail'/'not_found'.
    """
    # ── 1. Загрузка библиотеки ─────────────────────────────────────────────────
    if not library_json.exists():
        msg = (f"Residue library не найдена: {library_json}\n"
               "Запусти build_mania_residue_library_fixed.py для генерации.")
        if fail_on_error:
            raise FileNotFoundError(msg)
        print(f"[QC] WARNING: {msg}")
        return {}

    with open(library_json) as f:
        lib = json.load(f)

    lib_residues = lib.get('residues', {})
    lib_patches  = lib.get('patches',  {})

    # ── 2. Сбор всех resnames из всех условий ─────────────────────────────────
    system_resnames: dict = {}  # resname → set of conditions
    for condition, u in universe_dict.items():
        for res in u.atoms.residues:
            rn = res.resname.strip()
            system_resnames.setdefault(rn, set()).add(condition)

    # ── 3. Валидация ─────────────────────────────────────────────────────────
    # Категории остатков, которые не требуют строгой проверки
    SKIP_RESNAMES = {'TIP3','TP3M','HOH','WAT','SOD','POT','CLA','CAL',
                     'MG','ZN2','CES','Na+','Cl-','K+','Ca+','ZN'}

    results = {}
    ok_list, warn_list, fail_list, patch_list, skip_list = [], [], [], [], []

    for resname in sorted(system_resnames.keys()):
        if resname in SKIP_RESNAMES:
            results[resname] = {'status': 'skip', 'coverage': None, 'block_type': 'solvent/ion'}
            skip_list.append(resname)
            continue

        if resname in lib_residues:
            entry = lib_residues[resname]
            cov_str = entry.get('metadata', {})
            # Простая проверка: есть ли атомы в записи
            n_atoms = entry.get('metadata', {}).get('atom_count', len(entry.get('atoms', [])))
            if n_atoms > 0:
                status = 'ok'
                coverage = 1.0
                ok_list.append(resname)
            else:
                status = 'warning'
                coverage = 0.0
                warn_list.append(resname)
            results[resname] = {'status': status, 'coverage': coverage,
                                 'block_type': 'RESI',
                                 'source_file': entry.get('source_file', ''),
                                 'conditions': sorted(system_resnames[resname])}
        elif resname in lib_patches:
            results[resname] = {'status': 'patch_only', 'coverage': None,
                                  'block_type': 'PRES',
                                  'source_file': lib_patches[resname].get('source_file', ''),
                                  'conditions': sorted(system_resnames[resname])}
            patch_list.append(resname)
        else:
            results[resname] = {'status': 'not_found', 'coverage': 0.0,
                                  'block_type': 'MISSING',
                                  'source_file': '',
                                  'conditions': sorted(system_resnames[resname])}
            fail_list.append(resname)

    # ── 4. Вывод сводки ──────────────────────────────────────────────────────
    print("=" * 60)
    print("[QC] Residue Library Validation — MANIA v1.1")
    print("=" * 60)
    print(f"  Библиотека: {library_json}")
    print(f"  Residues в системе: {len(system_resnames)} уникальных")
    print()
    if ok_list:
        print(f"  ✓ ok ({len(ok_list)}): {', '.join(ok_list)}")
    if warn_list:
        print(f"  ⚠ warning — нет атомов в библиотеке ({len(warn_list)}): {', '.join(warn_list)}")
    if patch_list:
        print(f"  ℹ patch_only (PRES, не RESI) ({len(patch_list)}): {', '.join(patch_list)}")
    if fail_list:
        print(f"  ✗ NOT FOUND ({len(fail_list)}): {', '.join(fail_list)}")
    if skip_list:
        print(f"  – skip (solvent/ion) ({len(skip_list)}): {', '.join(skip_list)}")
    print("=" * 60)

    # ── 5. Сохранение QC-репорта ─────────────────────────────────────────────
    qc_rows = []
    for resname, info in results.items():
        qc_rows.append({
            'resname':    resname,
            'status':     info['status'],
            'coverage':   info.get('coverage', ''),
            'block_type': info.get('block_type', ''),
            'source_file':info.get('source_file', ''),
            'conditions': ','.join(info.get('conditions', [])),
        })
    pd.DataFrame(qc_rows).to_csv(export_dir / 'residue_qc_report.csv', index=False)
    print(f"  → residue_qc_report.csv сохранён ({len(qc_rows)} строк)")

    # ── 6. Аварийная остановка при fail_on_error ──────────────────────────────
    if fail_on_error and fail_list:
        raise RuntimeError(
            f"[QC] ОШИБКА: {len(fail_list)} residue(s) не найдены в библиотеке: "
            f"{fail_list}\n"
            "Добавь их в топологические файлы или используй normalize_residue_names.py"
        )
    return results

## Step 4. Trajectory loading (MDAnalysis Universe)

In [ ]:
import time

def _yadisk_download(token, yadisk_path, dest):
    import yadisk
    if dest.exists():
        print(f'  [cache] {dest.name}')
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    with yadisk.Client(token=token) as client:
        client.download(yadisk_path, str(dest))
    print(f'  [ok] {dest.name} {dest.stat().st_size/1024**2:.0f} MB {time.time()-t0:.1f}s')
    return dest

DATA_CACHE = OUTPUT_DIR / 'trajectory_cache'
DATA_CACHE.mkdir(exist_ok=True)

CONFIG = {}

if SOURCE == 'local':
    for cond, cfg in LOCAL_CONFIG.items():
        CONFIG[cond] = {
            'topology':   Path(cfg['topology']),
            'trajectory': Path(cfg['trajectory']),
            'label': cfg['label'],
            'ph':    cfg['ph'],
        }

elif SOURCE == 'yadisk':
    if not MW_TOKEN:
        try:
            from google.colab import userdata
            MW_TOKEN = userdata.get('MW_TOKEN')
            print('Токен загружен из Colab Secrets.')
        except Exception:
            raise ValueError('Укажи MW_TOKEN вручную.')
    import yadisk as _yd
    with _yd.Client(token=MW_TOKEN) as c:
        info = c.get_disk_info()
        print(f'Яндекс Диск: OK, свободно {(info.total_space - info.used_space)/1024**3:.1f} GB')
    for cond, cfg in YADISK_PATHS.items():
        topo = _yadisk_download(MW_TOKEN, cfg['topology'],
                                DATA_CACHE / cond / cfg['topology'].split('/')[-1])
        traj = _yadisk_download(MW_TOKEN, cfg['trajectory'],
                                DATA_CACHE / cond / cfg['trajectory'].split('/')[-1])
        CONFIG[cond] = {'topology': topo, 'trajectory': traj,
                        'label': cfg['label'], 'ph': cfg['ph']}

elif SOURCE == 'gdrive':
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')
    GDRIVE_BASE = Path('/content/drive/MyDrive/MANIA_data')
    for cond, (label, ph) in [('normal', (0, 7.4)), ('tumor', (1, 6.8))]:
        CONFIG[cond] = {
            'topology':   GDRIVE_BASE / cond / 'step7_production_10ns.tpr',
            'trajectory': GDRIVE_BASE / cond / 'trajectory.xtc',
            'label': label, 'ph': ph,
        }

UNIVERSES = {}
for condition, cfg in CONFIG.items():
    print(f'[{condition}] загружаю universe...')
    u = mda.Universe(str(cfg['topology']), str(cfg['trajectory']))
    UNIVERSES[condition] = u
    print(f'  frames={len(u.trajectory)} atoms={u.atoms.n_atoms} '
          f'protein_residues={u.select_atoms(PROTEIN_SELECTION).n_residues}')

Яндекс Диск: OK, свободно 686.3 GB
  [ok] step7_production_10ns.tpr 16 MB 7.6s
  [ok] step_full_0_30ns.xtc 5937 MB 476.9s
  [ok] step7_production_10ns.tpr 15 MB 8.5s
  [ok] step_full_0_30ns.xtc 5673 MB 449.3s
[normal] загружаю universe...
  frames=3003 atoms=551462 protein_residues=690
[tumor] загружаю universe...
  frames=3003 atoms=526846 protein_residues=690


## Step 4b. Residue Library QC  *(NEW in v1.1)*

Проверяет, что все residue names в системе есть в `mania_residue_library.json`
и имеют приемлемое покрытие атомных имён.

| Статус | Цвет | Значение |
|--------|------|----------|
| `ok` | ✓ | coverage = 1.0, все атомы совпали |
| `warning` | ⚠ | coverage < 1.0, частичное совпадение |
| `fail` | ✗ | coverage = 0 или residue не найден |
| `patch_only` | ℹ | совпадение только через PRES (patch), не RESI |

Если `RESIDUE_LIB_FAIL_ON_ERROR = True`, шаг бросает `RuntimeError` при наличии `fail`/`not_found`.


In [ ]:
# ── Запуск QC после загрузки траекторий (v1.1) ───────────────────────────────
_yadisk_download(MW_TOKEN, YADISK_LIB_PATH, RESIDUE_LIB_PATH)

if RESIDUE_LIB_VALIDATE:
    QC_RESULTS = run_residue_library_qc(
        UNIVERSES, RESIDUE_LIB_PATH, EXPORT_DIR,
        fail_on_error=RESIDUE_LIB_FAIL_ON_ERROR
    )

  [ok] mania_residue_library.json 33 MB 10.1s
[QC] Residue Library Validation — MANIA v1.1
  Библиотека: /content/mania_outputs/mania_exports/mania_residue_library.json
  Residues в системе: 39 уникальных

  ✓ ok (36): AFUC, ALA, ANE5AC, ARG, ASN, ASP, BGAL, BGALNA, BGLC, BGLCNA, BMAN, CER160, CHL1, CYS, GLN, GLU, GLY, HSD, HSP, ILE, LEU, LYS, MET, PHE, POPA, POPC, POPE, POPI, POPS, PRO, PSM, SER, THR, TRP, TYR, VAL
  – skip (solvent/ion) (3): CLA, SOD, TIP3
  → residue_qc_report.csv сохранён (39 строк)


## Step 5. ESM-2 embeddings (optional, USE_ESM2=False)
Если `USE_ESM2 = False`, узлы белка используют только структурные признаки (RMSF, SASA, DSSP, Φ/Ψ, TM-Z, Rg).


In [ ]:
USE_ESM2   = False  # ← True для ST-GNN
USE_HETERO = False  # True — только если нужен ST-GNN / PyG pipeline

ESM_CACHE_DIR = OUTPUT_DIR / 'esm_cache'
ESM_CACHE_DIR.mkdir(exist_ok=True)

ESM_DATA = {}

if USE_ESM2:
    import torch, esm
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
    model = model.eval().to(device)
    batch_converter = alphabet.get_batch_converter()

    def compute_esm2_embeddings(sequence):
        _, _, tokens = batch_converter([('protein', sequence)])
        tokens = tokens.to(device)
        with torch.no_grad():
            out = model(tokens, repr_layers=[33])
        return out['representations'][33][0, 1:len(sequence)+1].cpu().numpy().astype('float32')

    def _get_sequence(u):
        residues = u.select_atoms(PROTEIN_SELECTION).residues
        seq, resids, resnames = [], [], []
        for r in residues:
            aa = AA3_TO_1.get(r.resname)
            if aa:
                seq.append(aa); resids.append(int(r.resid)); resnames.append(r.resname)
        return ''.join(seq), resids, resnames

    for condition, u in UNIVERSES.items():
        seq, resid_list, resname_list = _get_sequence(u)
        cache_np   = ESM_CACHE_DIR / f'{condition}_esm2.npy'
        cache_meta = ESM_CACHE_DIR / f'{condition}_esm2_meta.json'
        if cache_np.exists() and json.loads(cache_meta.read_text()).get('sequence') == seq:
            emb = np.load(cache_np)
        else:
            emb = compute_esm2_embeddings(seq)
            np.save(cache_np, emb)
            cache_meta.write_text(json.dumps({'condition': condition,
                                               'sequence': seq, 'length': len(seq),
                                               'embedding_dim': int(emb.shape[1])}, indent=2))
        ESM_DATA[condition] = {'sequence': seq, 'resid_list': resid_list,
                                'resname_list': resname_list, 'embeddings': emb}
        print(f'[{condition}] ESM-2 embeddings shape = {emb.shape}')
else:
    for condition, u in UNIVERSES.items():
        residues     = u.select_atoms(PROTEIN_SELECTION).residues
        resid_list   = [int(r.resid)   for r in residues]
        resname_list = [r.resname       for r in residues]
        seq          = ''.join([AA3_TO_1.get(r.resname, 'X') for r in residues])
        ESM_DATA[condition] = {'sequence': seq, 'resid_list': resid_list,
                                'resname_list': resname_list, 'embeddings': None}
        print(f'[{condition}] ESM-2 пропущено. Residues: {len(resid_list)}')


[normal] ESM-2 пропущено. Residues: 690
[tumor] ESM-2 пропущено. Residues: 690


## Step 6. Structural features (RMSF, SASA, DSSP, Φ/Ψ, TM-Z, Rg(t))

**v1.1:** добавлен `compute_rg_timeseries()` — глобальный радиус инерции по белку.

- `rg_timeseries` (shape `[n_frames]`) — per-frame Rg в ångström
- `rg_mean`, `rg_std` — среднее и стандартное отклонение по траектории
- Используется как graph-level признак (не per-residue) для описания кластеров и сравнения условий


In [ ]:
DSSP_STATES  = ['H','B','E','G','I','T','S','C']
DSSP_TO_INDEX = {s:i for i,s in enumerate(DSSP_STATES)}

def align_feature_to_resids(arr, feature_resids, target_resids):
    idx = {int(r):i for i,r in enumerate(feature_resids)}
    out = np.zeros((len(target_resids), arr.shape[1]), dtype='float32')
    for j,r in enumerate(target_resids):
        if int(r) in idx: out[j] = arr[idx[int(r)]]
    return out

def compute_rmsf(u, stride=1):
    ca   = u.select_atoms(CA_SELECTION)
    calc = rms.RMSF(ca).run(step=stride)
    return calc.results.rmsf.reshape(-1,1).astype('float32'), [int(a.resid) for a in ca]

def compute_sasa(u, stride=50):
    import freesasa
    residues = u.select_atoms(PROTEIN_SELECTION).residues
    resids   = [int(r.resid) for r in residues]
    accum    = np.zeros(len(residues), dtype='float64')
    n        = 0
    tmp      = OUTPUT_DIR / '_tmp_sasa.pdb'
    protein  = u.select_atoms(PROTEIN_SELECTION)
    for fi in tqdm(range(0, len(u.trajectory), max(stride,1)), desc='SASA', leave=False):
        u.trajectory[fi]; protein.write(str(tmp))
        try:
            struct = freesasa.Structure(str(tmp))
            res    = freesasa.calc(struct)
            try:    areas = res.residueAreas()
            except: areas = freesasa.residueAreas(res, struct)
            by_r = {}
            for ch, cm in areas.items():
                for rs, ao in cm.items():
                    try:
                        rid = int(''.join(filter(str.isdigit, rs)))
                        by_r[rid] = float(ao.total)
                    except: pass
            for i,rid in enumerate(resids): accum[i] += by_r.get(int(rid),0.)
            n += 1
        except Exception as e: print(f'  frame {fi}: {e}')
    try: tmp.unlink()
    except: pass
    if n==0: return np.zeros((len(residues),1),'float32'), resids
    return (accum/n).reshape(-1,1).astype('float32'), resids

def compute_dssp(u, stride=1):
    residues = u.select_atoms(PROTEIN_SELECTION).residues
    resids   = [int(r.resid) for r in residues]
    onehot   = np.zeros((len(residues), len(DSSP_STATES)), 'float32')
    done     = False
    try:
        from MDAnalysis.analysis.dssp import DSSP
        ag = u.select_atoms(PROTEIN_SELECTION)
        d  = DSSP(ag).run(step=stride)
        if hasattr(d.results,'dssp'):
            arr = np.asarray(d.results.dssp)
            if arr.ndim==2:
                for col in range(min(arr.shape[1], len(residues))):
                    vals, cnts = np.unique(arr[:,col], return_counts=True)
                    state = vals[np.argmax(cnts)]
                    onehot[col, DSSP_TO_INDEX.get(state, DSSP_TO_INDEX['C'])] = 1.
                done = True
    except Exception as e: print(f'  DSSP primary: {e}')
    if not done:
        try:
            rama = dihedrals.Ramachandran(residues).run(step=stride)
            ang  = np.nanmean(rama.results.angles, axis=0) if rama.results.angles.ndim==3 else None
            if ang is not None:
                for i,(phi,psi) in enumerate(ang):
                    state = ('H' if -160<=phi<=-20 and -90<=psi<=45 else
                             'E' if -180<=phi<=-40 and 90<=psi<=180 else 'C')
                    onehot[i, DSSP_TO_INDEX[state]] = 1.
            else: onehot[:, DSSP_TO_INDEX['C']] = 1.
        except: onehot[:, DSSP_TO_INDEX['C']] = 1.
    return onehot, resids

def compute_phi_psi(u, stride=1):
    residues = u.select_atoms(PROTEIN_SELECTION).residues
    resids   = [int(r.resid) for r in residues]
    out      = np.zeros((len(residues),4),'float32')
    try:
        rama = dihedrals.Ramachandran(residues).run(step=stride)
        if rama.results.angles.ndim==3:
            ang = np.nanmean(rama.results.angles, axis=0)
            phi, psi = ang[:,0], ang[:,1]
            out[:,0] = np.nan_to_num(np.sin(np.deg2rad(phi)))
            out[:,1] = np.nan_to_num(np.cos(np.deg2rad(phi)))
            out[:,2] = np.nan_to_num(np.sin(np.deg2rad(psi)))
            out[:,3] = np.nan_to_num(np.cos(np.deg2rad(psi)))
    except Exception as e: print(f'  Phi/Psi fallback: {e}')
    return out, resids

def compute_tm_z(u):
    residues = u.select_atoms(PROTEIN_SELECTION).residues
    resids   = [int(r.resid) for r in residues]
    vals     = np.zeros((len(residues),1),'float32')
    try:
        refz = u.select_atoms(PROTEIN_SELECTION).center_of_mass()[2]
        for i,res in enumerate(residues):
            vals[i,0] = float(res.atoms.center_of_mass()[2] - refz)
    except: pass
    return vals, resids

# ── Rg(t) — NEW in v1.1 ─────────────────────────────────────────────────────
def compute_rg_timeseries(u, selection='protein', stride=1):
    """
    Compute per-frame radius of gyration Rg(t) for the specified selection.

    Rg(t) = sqrt( sum_i m_i * |r_i(t) - r_COM(t)|^2 / sum_i m_i )

    Returns
    -------
    rg_arr    : np.ndarray, shape (n_frames,), float32, values in Ångström
    frame_arr : np.ndarray, shape (n_frames,), int32, frame indices
    """
    ag         = u.select_atoms(selection)
    masses     = ag.masses.astype('float64')
    total_mass = masses.sum()
    if total_mass == 0:
        return np.array([], dtype='float32'), np.array([], dtype='int32')

    frame_indices = list(range(0, len(u.trajectory), max(stride, 1)))
    rg_vals = np.empty(len(frame_indices), dtype='float32')

    for out_idx, fi in enumerate(tqdm(frame_indices, desc=f'Rg(t)', leave=False)):
        u.trajectory[fi]
        pos = ag.positions.astype('float64')                # (N, 3)
        com = (masses[:, None] * pos).sum(0) / total_mass   # (3,)
        sq  = np.sum((pos - com)**2, axis=1)                # (N,)
        rg_vals[out_idx] = float(np.sqrt((masses * sq).sum() / total_mass))

    return rg_vals, np.array(frame_indices, dtype='int32')


# ── Расчёт всех признаков ────────────────────────────────────────────────────
STRUCTURAL_FEATURES = {}

for condition, u in UNIVERSES.items():
    target = ESM_DATA[condition]['resid_list']
    sf     = FEATURE_CALC_PARAMS['frame_stride_features']
    ss     = FEATURE_CALC_PARAMS['sasa_frame_stride']
    print(f'[{condition}] вычисляю структурные признаки...')

    rg_arr, rg_frames = (compute_rg_timeseries(u, PROTEIN_SELECTION, sf)
                         if COMPUTE_RG else (np.array([]), np.array([])))

    STRUCTURAL_FEATURES[condition] = {
        'rmsf':          align_feature_to_resids(*compute_rmsf(u, sf),     target),
        'sasa':          align_feature_to_resids(*compute_sasa(u, ss),     target),
        'dssp':          align_feature_to_resids(*compute_dssp(u, sf),     target),
        'phi_psi':       align_feature_to_resids(*compute_phi_psi(u, sf),  target),
        'tm_relative_z': align_feature_to_resids(*compute_tm_z(u),         target),
        # Rg — глобальные (не per-residue), хранятся отдельно
        'rg_timeseries': rg_arr,
        'rg_frames':     rg_frames,
        'rg_mean':       float(rg_arr.mean()) if len(rg_arr) > 0 else float('nan'),
        'rg_std':        float(rg_arr.std(ddof=1)) if len(rg_arr) > 1 else 0.0,
    }

    shape_report = {k: (v.shape if hasattr(v,'shape') else type(v).__name__)
                    for k,v in STRUCTURAL_FEATURES[condition].items()}
    print(f'{condition}', shape_report)
    if COMPUTE_RG:
        print(f'  Rg: mean={STRUCTURAL_FEATURES[condition]["rg_mean"]:.3f} Å  '
              f'std={STRUCTURAL_FEATURES[condition]["rg_std"]:.3f} Å  '
              f'frames={len(rg_arr)}')


[normal] вычисляю структурные признаки...


Rg(t):   0%|          | 0/301 [00:00<?, ?it/s]

SASA:   0%|          | 0/61 [00:00<?, ?it/s]

  Phi/Psi fallback: could not broadcast input array from shape (688,) into shape (690,)
normal {'rmsf': (690, 1), 'sasa': (690, 1), 'dssp': (690, 8), 'phi_psi': (690, 4), 'tm_relative_z': (690, 1), 'rg_timeseries': (301,), 'rg_frames': (301,), 'rg_mean': 'float', 'rg_std': 'float'}
  Rg: mean=36.438 Å  std=1.269 Å  frames=301
[tumor] вычисляю структурные признаки...


Rg(t):   0%|          | 0/301 [00:00<?, ?it/s]

SASA:   0%|          | 0/61 [00:00<?, ?it/s]

  Phi/Psi fallback: could not broadcast input array from shape (688,) into shape (690,)
tumor {'rmsf': (690, 1), 'sasa': (690, 1), 'dssp': (690, 8), 'phi_psi': (690, 4), 'tm_relative_z': (690, 1), 'rg_timeseries': (301,), 'rg_frames': (301,), 'rg_mean': 'float', 'rg_std': 'float'}
  Rg: mean=36.290 Å  std=0.667 Å  frames=301


## Step 7. Protein–protein contacts (per-condition)

**v1.2**:
- Введён `buildatomcache()`: все атомные выборки (доноры, акцепторы, водороды,
  тяжёлые атомы, кольца, катионы, заряженные группы) готовятся один раз
  до цикла по фреймам. Это устраняет повторные `selectatoms` на каждый кадр
  и сокращает время расчёта.
- H-bonds: проверка угла D–H···A по реальному атому H из топологии
  (через `atom.bonds`), а не геометрически по позиции донора.
- Aromatic π–π: нормаль кольца вычисляется через SVD; проверяются
  параллельная (angle < parallel_max) и T-образная (60° < angle < 120°) ориентации.
- Cation–π: добавлена проверка угла катион → нормаль кольца.
- Ionic / salt bridge: объединены в единую `frameioniccache()` с переключением
  через `cutoffkey`.

`CONTACT_MAPS[condition]` — словарь `{edge_type: {edge_index, edge_attr}}`.
Backbone считается отдельно (не через `InteractionAccumulator`).

In [ ]:
from MDAnalysis.lib.distances import capped_distance, calc_angles
import numpy as np
from collections import defaultdict


# ── InteractionAccumulator ──────────────────────────────────────────────────

class InteractionAccumulator:
    """Accumulates per-frame contacts → contact_freq, mean_dist, std_dist."""

    def __init__(self):
        self._counts  = defaultdict(int)    # (i,j) → n frames present
        self._dists   = defaultdict(list)   # (i,j) → [dist per frame]
        self._n_frames = 0

    def tick(self):
        self._n_frames += 1

    def add(self, i: int, j: int, dist: float):
        key = (min(i, j), max(i, j))
        self._counts[key] += 1
        self._dists[key].append(dist)

    def finalize(self, edge_type: str, min_freq: float = 0.0):
        rows = []
        if self._n_frames == 0:
            return rows
        for (i, j), cnt in self._counts.items():
            freq = cnt / self._n_frames
            if freq < min_freq:
                continue
            dists = self._dists[(i, j)]
            rows.append({
                'resid_i':     i,
                'resid_j':     j,
                'edge_type':   edge_type,
                'contact_freq': round(freq, 4),
                'mean_dist_A': round(float(np.mean(dists)), 3),
                'std_dist_A':  round(float(np.std(dists)),  3),
            })
        return rows


# ── Per-frame interaction detectors ────────────────────────────────────────

def frame_backbone(acc, rmap, ca_positions, resids_ordered, p):
    """Sequential Cα i↔i+1 — всегда добавляется, dist < backbone_max_ca_dist_A."""
    cutoff = p.get('backbone_max_ca_dist_A', BACKBONE_MAX_CA_DIST_A)
    for k in range(len(resids_ordered) - 1):
        ri = resids_ordered[k]
        rj = resids_ordered[k + 1]
        if rj - ri != 1:           # пропуск в нумерации — не backbone
            continue
        if ri not in rmap or rj not in rmap:
            continue
        d = float(np.linalg.norm(ca_positions[k + 1] - ca_positions[k]))
        if d <= cutoff:
            acc.add(ri, rj, d)


def frame_hbond(acc, rmap, u, p):
    """Backbone + sidechain H-bonds через атомную геометрию."""
    protein       = u.select_atoms('protein')
    donors_sel    = protein.select_atoms('name N O and not resname PRO')
    acceptors_sel = protein.select_atoms('name O N')

    if len(donors_sel) == 0 or len(acceptors_sel) == 0:
        return

    cutoff    = p.get('hbond_distance_cutoff_A', 3.5)
    angle_min = p.get('hbond_angle_cutoff_deg', 120.0)

    pairs, dists = capped_distance(
        donors_sel.positions,
        acceptors_sel.positions,
        max_cutoff=cutoff,
        return_distances=True,
        box=u.dimensions,
    )

    for (i_d, i_a), d in zip(pairs, dists):
        don_atom = donors_sel[i_d]
        acc_atom = acceptors_sel[i_a]
        ri = int(don_atom.resid)
        rj = int(acc_atom.resid)
        if ri == rj or ri not in rmap or rj not in rmap:
            continue

        # ── ищем H через bonds (не через select_atoms!) ────────────
        h_pos = None
        try:
            for bond in don_atom.bonds:
                partner = bond.partner(don_atom)
                if partner.name.startswith('H'):
                    h_pos = partner.position.copy()
                    break
        except Exception:
            pass

        if h_pos is not None:
            angle = float(np.degrees(
                calc_angles(don_atom.position, h_pos, acc_atom.position)
            ))
            if angle < angle_min:
                continue
        # если H нет в топологии — принимаем по дистанции (fallback)

        acc.add(ri, rj, float(d))


def frame_vdw(acc, rmap, heavy_pos, heavy_resids, p):
    dmin = p.get('vdw_distance_min_A', 3.0)
    dmax = p.get('vdw_distance_max_A', 4.5)
    pairs, dists = capped_distance(
        heavy_pos, heavy_pos, max_cutoff=dmax, return_distances=True
    )
    for (i, j), d in zip(pairs, dists):
        if i >= j:
            continue
        if d < dmin:
            continue
        ri, rj = heavy_resids[i], heavy_resids[j]
        if ri == rj:
            continue
        if ri not in rmap or rj not in rmap:
            continue
        acc.add(ri, rj, float(d))


def frame_hydrophobic(acc, rmap, cb_pos, cb_resids, cb_resnames, p):
    HYDROPHOBIC = {'ALA', 'VAL', 'LEU', 'ILE', 'MET', 'PHE', 'TRP', 'PRO', 'CYS'}
    cutoff = p.get('hydrophobic_cutoff_A', 5.0)
    mask = [rn in HYDROPHOBIC for rn in cb_resnames]
    if sum(mask) < 2:
        return
    idx = np.where(mask)[0]
    pos_h = cb_pos[idx]
    res_h = [cb_resids[i] for i in idx]
    pairs, dists = capped_distance(pos_h, pos_h, max_cutoff=cutoff, return_distances=True)
    for (i, j), d in zip(pairs, dists):
        if i >= j:
            continue
        ri, rj = res_h[i], res_h[j]
        if ri == rj:
            continue
        if ri not in rmap or rj not in rmap:
            continue
        acc.add(ri, rj, float(d))


def frame_aromatic_pi(acc, rmap, u, p):
    AROMATIC = {'PHE': 'CG CD1 CD2 CE1 CE2 CZ',
                'TYR': 'CG CD1 CD2 CE1 CE2 CZ',
                'TRP': 'CG CD1 CD2 NE1 CE2 CE3 CZ2 CZ3 CH2',
                'HIS': 'CG ND1 CD2 CE1 NE2',
                'HSD': 'CG ND1 CD2 CE1 NE2',
                'HSP': 'CG ND1 CD2 CE1 NE2'}
    cutoff_c = p.get('aromatic_pi_centroid_cutoff_A', 7.0)
    angle_par = p.get('aromatic_parallel_angle_max_deg', 30.0)
    angle_t_min = p.get('aromatic_tshape_angle_min_deg', 60.0)
    angle_t_max = p.get('aromatic_tshape_angle_max_deg', 120.0)

    rings = []
    for resname, ring_atoms in AROMATIC.items():
        sel = u.select_atoms(f'protein and resname {resname} and name {ring_atoms}')
        for res in sel.residues:
            ra = res.atoms.select_atoms(f'name {ring_atoms}')
            if len(ra) >= 5:
                centroid = ra.center_of_geometry()
                # нормаль к плоскости кольца через SVD
                coords = ra.positions - centroid
                _, _, vh = np.linalg.svd(coords)
                normal = vh[-1]
                rings.append((int(res.resid), centroid, normal))

    for i in range(len(rings)):
        for j in range(i + 1, len(rings)):
            ri, ci, ni = rings[i]
            rj, cj, nj = rings[j]
            d = float(np.linalg.norm(ci - cj))
            if d > cutoff_c:
                continue
            if ri not in rmap or rj not in rmap:
                continue
            cos_a = abs(float(np.dot(ni, nj) /
                        (np.linalg.norm(ni) * np.linalg.norm(nj) + 1e-9)))
            angle = float(np.degrees(np.arccos(np.clip(cos_a, 0, 1))))
            if angle <= angle_par or angle_t_min <= angle <= angle_t_max:
                acc.add(ri, rj, d)


def frame_ionic(acc, rmap, u, p):
    POS = {'LYS': 'NZ', 'ARG': 'CZ', 'HIS': 'ND1', 'HSP': 'ND1', 'HSD': 'ND1'}
    NEG = {'ASP': 'CG', 'GLU': 'CD'}
    cutoff = p.get('ionic_cutoff_A', 6.0)
    pos_atoms, pos_resids = [], []
    neg_atoms, neg_resids = [], []
    for rn, an in POS.items():
        sel = u.select_atoms(f'protein and resname {rn} and name {an}')
        pos_atoms.append(sel.positions if len(sel) else np.zeros((0, 3)))
        pos_resids.extend([int(a.resid) for a in sel])
    for rn, an in NEG.items():
        sel = u.select_atoms(f'protein and resname {rn} and name {an}')
        neg_atoms.append(sel.positions if len(sel) else np.zeros((0, 3)))
        neg_resids.extend([int(a.resid) for a in sel])
    if not pos_resids or not neg_resids:
        return
    pos_pos = np.vstack([a for a in pos_atoms if len(a)])
    neg_pos = np.vstack([a for a in neg_atoms if len(a)])
    pairs, dists = capped_distance(pos_pos, neg_pos, max_cutoff=cutoff, return_distances=True)
    for (i, j), d in zip(pairs, dists):
        ri, rj = pos_resids[i], neg_resids[j]
        if ri == rj:
            continue
        if ri not in rmap or rj not in rmap:
            continue
        acc.add(ri, rj, float(d))


def frame_salt_bridge(acc, rmap, u, p):
    """Как ionic, но более строгий cutoff (salt_bridge_cutoff_A)."""
    POS = {'LYS': 'NZ', 'ARG': 'NH1 NH2'}
    NEG = {'ASP': 'OD1 OD2', 'GLU': 'OE1 OE2'}
    cutoff = p.get('salt_bridge_cutoff_A', 4.0)
    pos_atoms, pos_resids = [], []
    neg_atoms, neg_resids = [], []
    for rn, names in POS.items():
        sel = u.select_atoms(f'protein and resname {rn} and name {names}')
        for a in sel:
            pos_atoms.append(a.position)
            pos_resids.append(int(a.resid))
    for rn, names in NEG.items():
        sel = u.select_atoms(f'protein and resname {rn} and name {names}')
        for a in sel:
            neg_atoms.append(a.position)
            neg_resids.append(int(a.resid))
    if not pos_atoms or not neg_atoms:
        return
    pos_pos = np.array(pos_atoms)
    neg_pos = np.array(neg_atoms)
    pairs, dists = capped_distance(pos_pos, neg_pos, max_cutoff=cutoff, return_distances=True)
    for (i, j), d in zip(pairs, dists):
        ri, rj = pos_resids[i], neg_resids[j]
        if ri == rj:
            continue
        if ri not in rmap or rj not in rmap:
            continue
        acc.add(ri, rj, float(d))


def frame_cation_pi(acc, rmap, u, p):
    AROMATIC = {'PHE': 'CG CD1 CD2 CE1 CE2 CZ',
                'TYR': 'CG CD1 CD2 CE1 CE2 CZ',
                'TRP': 'CG CD1 CD2 NE1 CE2 CE3 CZ2 CZ3 CH2',
                'HIS': 'CG ND1 CD2 CE1 NE2', 'HSD': 'CG ND1 CD2 CE1 NE2'}
    CATION = {'LYS': 'NZ', 'ARG': 'CZ'}
    cutoff = p.get('cation_pi_cutoff_A', 6.0)
    rings = []
    for rn, an in AROMATIC.items():
        sel = u.select_atoms(f'protein and resname {rn} and name {an}')
        for res in sel.residues:
            ra = res.atoms.select_atoms(f'name {an}')
            if len(ra) >= 4:
                rings.append((int(res.resid), ra.center_of_geometry()))
    cations = []
    for rn, an in CATION.items():
        sel = u.select_atoms(f'protein and resname {rn} and name {an}')
        for a in sel:
            cations.append((int(a.resid), a.position))
    for ri, cpos in cations:
        for rj, rpos in rings:
            if ri == rj:
                continue
            d = float(np.linalg.norm(cpos - rpos))
            if d <= cutoff:
                if ri not in rmap or rj not in rmap:
                    continue
                acc.add(ri, rj, d)


# ── Главная функция Step 7 ─────────────────────────────────────────────────

def build_protein_contact_maps(universes, feature_calc_params,
                                conditions=None, export_dir=None,
                                min_contact_freq=None):
    """
    Строит protein-protein contact maps для каждого условия.
    Возвращает dict[cond -> pd.DataFrame] с колонками:
        resid_i, resid_j, edge_type, contact_freq, mean_dist_A, std_dist_A
    """
    if conditions is None:
        conditions = list(universes.keys())
    if min_contact_freq is None:
        min_contact_freq = feature_calc_params.get('min_contact_frequency', 0.30)

    p = feature_calc_params
    stride = p.get('frame_stride_contacts', 10)

    all_edges = {}

    for cond in conditions:
        u = universes[cond]
        protein = u.select_atoms('protein')
        ca_ag = u.select_atoms('protein and name CA')

        # rmap: resid -> resid (для валидации присутствия в белке)
        rmap = {int(r.resid): int(r.resid) for r in protein.residues}
        resids_ordered = sorted(rmap.keys())

        # Предвычисляем CB-позиции (Cβ, для Gly — Cα)
        cb_atoms, cb_resids, cb_resnames = [], [], []
        for res in protein.residues:
            cb = res.atoms.select_atoms('name CB')
            if len(cb) == 0:
                cb = res.atoms.select_atoms('name CA')
            if len(cb) > 0:
                cb_atoms.append(cb[0])
                cb_resids.append(int(res.resid))
                cb_resnames.append(res.resname)

        # Предвычисляем heavy atoms для vdw
        heavy_ag = protein.select_atoms('not name H*')

        # Accumulators для каждого типа
        stores = {
            'backbone':    InteractionAccumulator(),
            'hbond':       InteractionAccumulator(),
            'disulfide':   InteractionAccumulator(),
            'vdw':         InteractionAccumulator(),
            'hydrophobic': InteractionAccumulator(),
            'aromatic_pi': InteractionAccumulator(),
            'cation_pi':   InteractionAccumulator(),
            'ionic':       InteractionAccumulator(),
            'salt_bridge': InteractionAccumulator(),
        }

        frames_used = 0
        print(f'[{cond}] building contacts (stride={stride}, '
              f'frames={len(u.trajectory)}) ...')

        for ts in tqdm(u.trajectory[::stride], desc=f'contacts {cond}'):
            for acc in stores.values():
                acc.tick()

            # Backbone
            ca_positions = np.array([a.position for a in ca_ag])
            ca_resids    = [int(a.resid) for a in ca_ag]
            frame_backbone(stores['backbone'], rmap, ca_positions, ca_resids, p)

            # H-bonds (атомный уровень)
            frame_hbond(stores['hbond'], rmap, u, p)

            # Disulfide
            cys_sg = u.select_atoms('protein and resname CYS and name SG')
            if len(cys_sg) > 1:
                sg_pos    = cys_sg.positions
                sg_resids = [int(a.resid) for a in cys_sg]
                sg_cutoff = p.get('disulfide_sg_cutoff_A', 2.2)
                pairs_ss, dists_ss = capped_distance(
                    sg_pos, sg_pos, max_cutoff=sg_cutoff, return_distances=True)
                for (i, j), d in zip(pairs_ss, dists_ss):
                    if i >= j:
                        continue
                    ri, rj = sg_resids[i], sg_resids[j]
                    if ri not in rmap or rj not in rmap:
                        continue
                    stores['disulfide'].add(ri, rj, float(d))

            # VdW
            frame_vdw(stores['vdw'], rmap,
                      heavy_ag.positions,
                      [int(a.resid) for a in heavy_ag], p)

            # Hydrophobic
            cb_pos_now = np.array([a.position for a in cb_atoms])
            frame_hydrophobic(stores['hydrophobic'], rmap,
                              cb_pos_now, cb_resids, cb_resnames, p)

            # Aromatic pi
            frame_aromatic_pi(stores['aromatic_pi'], rmap, u, p)

            # Ionic
            frame_ionic(stores['ionic'], rmap, u, p)

            # Salt bridge
            frame_salt_bridge(stores['salt_bridge'], rmap, u, p)

            # Cation-pi
            frame_cation_pi(stores['cation_pi'], rmap, u, p)

            frames_used += 1

        # ── Финализация ─────────────────────────────────────────────────────
        all_rows = []
        for edge_type, acc in stores.items():
            rows = acc.finalize(edge_type, min_freq=min_contact_freq)
            all_rows.extend(rows)
            print(f'  [{cond}] {edge_type:12s}: {len(rows):5d} edges '
                  f'(freq >= {min_contact_freq})')

        edges_df = pd.DataFrame(all_rows)
        if len(edges_df):
            edges_df = edges_df[edges_df['resid_i'] < edges_df['resid_j']].copy()
            edges_df = edges_df.sort_values(
                ['edge_type', 'resid_i', 'resid_j']).reset_index(drop=True)

        all_edges[cond] = edges_df
        print(f'[{cond}] TOTAL: {len(edges_df)} edges, {frames_used} frames\n')

        # ── Сохранение ──────────────────────────────────────────────────────
        if export_dir is not None:
            out = Path(export_dir) / f'protein_contact_edges_undirected_{cond}.csv'
            edges_df.to_csv(out, index=False)
            print(f'  saved: {out.name}')

    return all_edges


# ── Запуск ─────────────────────────────────────────────────────────────────
CONTACT_MAPS = build_protein_contact_maps(
    universes=UNIVERSES,
    feature_calc_params=FEATURE_CALC_PARAMS,
    conditions=list(UNIVERSES.keys()),
    export_dir=EXPORT_DIR,
    min_contact_freq=FEATURE_CALC_PARAMS['min_contact_frequency'],
)

print('Step 7 done.')

[normal] building contacts (stride=10, frames=3003) ...


contacts normal:   0%|          | 0/301 [00:00<?, ?it/s]

  [normal] backbone    :   689 edges (freq >= 0.3)
  [normal] hbond       :  1693 edges (freq >= 0.3)
  [normal] disulfide   :     0 edges (freq >= 0.3)
  [normal] vdw         :  3263 edges (freq >= 0.3)
  [normal] hydrophobic :    84 edges (freq >= 0.3)
  [normal] aromatic_pi :    24 edges (freq >= 0.3)
  [normal] cation_pi   :     7 edges (freq >= 0.3)
  [normal] ionic       :    22 edges (freq >= 0.3)
  [normal] salt_bridge :    21 edges (freq >= 0.3)
[normal] TOTAL: 5803 edges, 301 frames

  saved: protein_contact_edges_undirected_normal.csv
[tumor] building contacts (stride=10, frames=3003) ...


contacts tumor:   0%|          | 0/301 [00:00<?, ?it/s]

  [tumor] backbone    :   689 edges (freq >= 0.3)
  [tumor] hbond       :  1716 edges (freq >= 0.3)
  [tumor] disulfide   :     0 edges (freq >= 0.3)
  [tumor] vdw         :  3330 edges (freq >= 0.3)
  [tumor] hydrophobic :    99 edges (freq >= 0.3)
  [tumor] aromatic_pi :    26 edges (freq >= 0.3)
  [tumor] cation_pi   :     6 edges (freq >= 0.3)
  [tumor] ionic       :    26 edges (freq >= 0.3)
  [tumor] salt_bridge :    22 edges (freq >= 0.3)
[tumor] TOTAL: 5914 edges, 301 frames

  saved: protein_contact_edges_undirected_tumor.csv
Step 7 done.


## Step 7b. Per-frame contact export (contacts_perframe parquet)

**v1.2**: backbone включён в пофреймовый экспорт.
Лог подсчёта детализирован по типам: backbone / hbond / vdw / hydrophobic.

In [ ]:
# ============================================================
# MANIA preprocessing — Step 7b: Per-frame contact export
# Package: mania/preprocessing/contacts_perframe.py
#
# ФИКСЫ:
#   - rmap: resid -> resid
#   - backbone: сохраняется per-frame
#   - hbond: distance + angle, H ищется через atom.bonds
#   - убран старый select_atoms(...bonded atom...)
# ============================================================

from MDAnalysis.lib.distances import capped_distance, calc_angles

_STRIDE = FEATURE_CALC_PARAMS['frame_stride_contacts']
_CUTOFFS = {
    'backbone': BACKBONE_MAX_CA_DIST_A,
    'hbond': FEATURE_CALC_PARAMS['hbond_distance_cutoff_A'],
    'vdw': FEATURE_CALC_PARAMS['vdw_distance_max_A'],
    'hydrophobic': FEATURE_CALC_PARAMS['hydrophobic_cutoff_A'],
}
_VDW_MIN = FEATURE_CALC_PARAMS['vdw_distance_min_A']
_HBOND_ANG = FEATURE_CALC_PARAMS['hbond_angle_cutoff_deg']
_HYDRO_AA = {'ALA', 'VAL', 'LEU', 'ILE', 'MET', 'PHE', 'TRP', 'PRO', 'TYR'}

for condition, u in UNIVERSES.items():

    protein = u.select_atoms(PROTEIN_SELECTION)
    rmap = {int(r.resid): int(r.resid) for r in protein.residues}

    ca_ag = u.select_atoms(CA_SELECTION)
    donor_ag = u.select_atoms('protein and (name N or name O) and not name O*')
    acceptor_ag = u.select_atoms('protein and (name O or name N)')
    heavy_ag = u.select_atoms('protein and not name H*')

    hydro_resids = []
    sidechain_ag = {}
    for res in protein.residues:
        if res.resname in _HYDRO_AA:
            sc = res.atoms.select_atoms('not backbone')
            if len(sc) > 0:
                hydro_resids.append(int(res.resid))
                sidechain_ag[int(res.resid)] = sc

    ca_res = [int(a.resid) for a in ca_ag]
    heavy_resids = [int(a.resid) for a in heavy_ag]

    rows = []
    frame_indices = list(range(0, len(u.trajectory), _STRIDE))

    for fi in tqdm(frame_indices, desc=f'per-frame {condition}', leave=True):
        u.trajectory[fi]

        # backbone: Cα(i) -- Cα(i+1)
        pairs, dd = capped_distance(
            ca_ag.positions,
            ca_ag.positions,
            max_cutoff=_CUTOFFS['backbone'],
            return_distances=True,
        )
        for (i, j), d in zip(pairs, dd):
            ri, rj = ca_res[i], ca_res[j]
            if i >= j:
                continue
            if ri >= rj or abs(ri - rj) != 1:
                continue
            if ri in rmap and rj in rmap:
                rows.append({
                    'frame': fi,
                    'resid_i': ri,
                    'resid_j': rj,
                    'edge_type': 'backbone',
                    'dist_A': round(float(d), 3),
                })

        # hbond: donor-acceptor distance + D-H...A angle
        if len(donor_ag) > 0 and len(acceptor_ag) > 0:
            pairs, dd = capped_distance(
                donor_ag.positions,
                acceptor_ag.positions,
                max_cutoff=_CUTOFFS['hbond'],
                return_distances=True,
            )
            for (i, j), d in zip(pairs, dd):
                don_atom = donor_ag[int(i)]
                acc_atom = acceptor_ag[int(j)]
                rd = int(don_atom.resid)
                ra = int(acc_atom.resid)

                if rd == ra or rd not in rmap or ra not in rmap:
                    continue

                h_pos = None
                try:
                    for bond in don_atom.bonds:
                        partner = bond.partner(don_atom)
                        if partner.name.startswith('H'):
                            h_pos = partner.position.copy()
                            break
                except Exception:
                    pass

                if h_pos is not None:
                    angle = float(np.degrees(
                        calc_angles(don_atom.position, h_pos, acc_atom.position)
                    ))
                    if angle < _HBOND_ANG:
                        continue

                rows.append({
                    'frame': fi,
                    'resid_i': min(rd, ra),
                    'resid_j': max(rd, ra),
                    'edge_type': 'hbond',
                    'dist_A': round(float(d), 3),
                })

        # vdw: heavy atom pairs in window
        if len(heavy_ag) > 0:
            pairs, dd = capped_distance(
                heavy_ag.positions,
                heavy_ag.positions,
                max_cutoff=_CUTOFFS['vdw'],
                return_distances=True,
            )
            for (i, j), d in zip(pairs, dd):
                if i >= j or d < _VDW_MIN:
                    continue
                ri = heavy_resids[int(i)]
                rj = heavy_resids[int(j)]
                if ri == rj or ri not in rmap or rj not in rmap:
                    continue
                rows.append({
                    'frame': fi,
                    'resid_i': min(ri, rj),
                    'resid_j': max(ri, rj),
                    'edge_type': 'vdw',
                    'dist_A': round(float(d), 3),
                })

        # hydrophobic: sidechain COM
        if len(hydro_resids) >= 2:
            centers = [
                (r, sidechain_ag[r].center_of_mass())
                for r in hydro_resids
                if len(sidechain_ag[r]) > 0
            ]
            if len(centers) >= 2:
                coords = np.array([c[1] for c in centers])
                pairs, dd = capped_distance(
                    coords,
                    coords,
                    max_cutoff=_CUTOFFS['hydrophobic'],
                    return_distances=True,
                )
                for (i, j), d in zip(pairs, dd):
                    if i >= j:
                        continue
                    ri, rj = centers[i][0], centers[j][0]
                    if ri == rj or ri not in rmap or rj not in rmap:
                        continue
                    rows.append({
                        'frame': fi,
                        'resid_i': min(ri, rj),
                        'resid_j': max(ri, rj),
                        'edge_type': 'hydrophobic',
                        'dist_A': round(float(d), 3),
                    })

    if rows:
        pf_df = pd.DataFrame(rows)

        bb_count = int((pf_df['edge_type'] == 'backbone').sum())
        hb_count = int((pf_df['edge_type'] == 'hbond').sum())
        vd_count = int((pf_df['edge_type'] == 'vdw').sum())
        hy_count = int((pf_df['edge_type'] == 'hydrophobic').sum())

        print(
            f'[7b] {condition}: {len(pf_df):,} contact-frames | '
            f'backbone={bb_count:,} | hbond={hb_count:,} | '
            f'vdw={vd_count:,} | hydrophobic={hy_count:,}'
        )

        schema = pa.schema([
            ('frame', pa.int32()),
            ('resid_i', pa.int32()),
            ('resid_j', pa.int32()),
            ('edge_type', pa.dictionary(pa.int8(), pa.string())),
            ('dist_A', pa.float32()),
        ])

        table = pa.Table.from_pandas(
            pf_df.astype({
                'frame': 'int32',
                'resid_i': 'int32',
                'resid_j': 'int32',
                'dist_A': 'float32',
            }),
            schema=schema,
            preserve_index=False,
        )

        out_pq = EXPORT_DIR / f'contacts_perframe_{condition}.parquet'
        pq.write_table(table, out_pq, compression='snappy')

        print(
            f'         saved → {out_pq.name} '
            f'({out_pq.stat().st_size / 1024**2:.1f} MB)'
        )
    else:
        print(f'[7b] {condition}: нет контактов')

per-frame normal:   0%|          | 0/301 [00:00<?, ?it/s]

[7b] normal: 6,590,357 contact-frames | backbone=207,389 | hbond=146,984 | vdw=6,211,272 | hydrophobic=24,712
         saved → contacts_perframe_normal.parquet (17.4 MB)


per-frame tumor:   0%|          | 0/301 [00:00<?, ?it/s]

[7b] tumor: 6,623,526 contact-frames | backbone=207,389 | hbond=148,114 | vdw=6,242,215 | hydrophobic=25,808
         saved → contacts_perframe_tumor.parquet (17.5 MB)


## Step 8. Cα alignment (Kabsch) & node coordinates

In [ ]:
def kabsch_align(P, Q):
    mu_P, mu_Q = P.mean(0), Q.mean(0)
    Pc, Qc = P - mu_P, Q - mu_Q
    H = Pc.T @ Qc
    U, S, Vt = np.linalg.svd(H)
    d = np.linalg.det(Vt.T @ U.T)
    D = np.diag([1, 1, d])
    R = Vt.T @ D @ U.T
    t = mu_Q - R @ mu_P
    P_aligned = (R @ P.T).T + t
    rmsd = float(np.sqrt(np.mean(np.sum((P_aligned - Q)**2, axis=1))))
    return P_aligned, R, t, rmsd

conditions  = list(UNIVERSES.keys())
ref_cond    = conditions[0]
ref_u       = UNIVERSES[ref_cond]
ref_u.trajectory[0]
ref_ca      = ref_u.select_atoms(CA_SELECTION)
ref_resids  = [int(a.resid) for a in ref_ca]
ref_pos     = ref_ca.positions.copy()

CA_ALIGNED       = {}
ALIGNMENT_RESULT = {}

for condition, u in UNIVERSES.items():
    u.trajectory[0]
    ca       = u.select_atoms(CA_SELECTION)
    ca_resids= [int(a.resid) for a in ca]
    common   = sorted(set(ref_resids) & set(ca_resids))
    ref_idx  = {r:i for i,r in enumerate(ref_resids)}
    cond_idx = {r:i for i,r in enumerate(ca_resids)}
    P = ca.positions[[cond_idx[r] for r in common]].copy()
    Q = ref_pos[[ref_idx[r] for r in common]].copy()
    P_aln, R, t, rmsd = kabsch_align(P, Q)
    all_pos  = (R @ ca.positions.T).T + t
    CA_ALIGNED[condition]       = all_pos
    ALIGNMENT_RESULT[condition] = {'rmsd': rmsd, 'R': R.tolist(), 't': t.tolist()}
    print(f'[ALN] {condition}: RMSD={rmsd:.4f} Å')

CA_COORDS = {}
for condition, u in UNIVERSES.items():
    ca = u.select_atoms(CA_SELECTION)
    CA_COORDS[condition] = {int(a.resid): a.position.tolist() for a in ca}
    print(f'{condition}: {len(CA_COORDS[condition])} Cα positions extracted')


[ALN] normal: RMSD=0.0000 Å
[ALN] tumor: RMSD=10.1222 Å
normal: 690 Cα positions extracted
tumor: 690 Cα positions extracted


## Step 9. Non-protein nodes (lipids, glycans, ligands)

In [ ]:
NPN_CUTOFFS = {
    'lipid':  FEATURE_CALC_PARAMS.get('protein_lipid_cutoff_A', 6.0),
    'glycan': FEATURE_CALC_PARAMS.get('protein_glycan_cutoff_A', 6.0),
    'ligand': FEATURE_CALC_PARAMS.get('protein_glycan_cutoff_A', 6.0),
}
NPN_TOP_K = 5

LIGAND_SEL = (
    "not (protein or resname " +
    " ".join(list(GLYCAN_RESNAMES) + LIPID_SELECTION.split()[1:]) +
    ") and not (resname WAT TIP3 SOL HOH) and not ion"
)

def _extract_entities(u, cond):
    rows = []
    for node_type, sel in [('lipid', LIPID_SELECTION),
                            ('glycan', 'resname ' + ' '.join(GLYCAN_RESNAMES)),
                            ('ligand', LIGAND_SEL)]:
        try:
            ag = u.select_atoms(sel)
        except Exception: continue
        if len(ag)==0: continue
        for res in ag.residues:
            com = res.atoms.center_of_mass()
            rows.append(dict(node_id=f'{node_type}_{res.segid}_{res.resid}',
                              node_type=node_type, entity_name=res.resname,
                              resid=int(res.resid), segid=str(res.segid),
                              x=float(com[0]), y=float(com[1]), z=float(com[2]),
                              n_heavy=int(len(res.atoms.select_atoms('not name H*'))),
                              condition=cond))
    return pd.DataFrame(rows)

def _nearest_contacts(u, npdf, ca_coords, protein_resids, cutoffs, top_k=5):
    rows = []
    if len(npdf)==0: return pd.DataFrame()
    for _, nrow in npdf.iterrows():
        cutoff = cutoffs.get(nrow['node_type'], 6.0)
        try:
            ag = u.select_atoms(f"resid {nrow['resid']} and not protein")
            if len(ag)==0: continue
            dists  = distance_array(ag.positions, ca_coords, box=None).min(axis=0)
            within = np.where(dists <= cutoff)[0]
            if len(within)==0: continue
            order  = within[np.argsort(dists[within])][:top_k]
            for rank, idx in enumerate(order, 1):
                rows.append(dict(np_node_id=nrow['node_id'], node_type=nrow['node_type'],
                                  entity_name=nrow['entity_name'],
                                  protein_resid=int(protein_resids[idx]),
                                  distance_A=float(dists[idx]), rank=rank,
                                  condition=nrow['condition']))
        except Exception as e: print(f'  [NPN] skip: {e}')
    return pd.DataFrame(rows)

NONPROTEIN_NODES = {}
NP_CONTACT_EDGES = {}

for condition, u in UNIVERSES.items():
    u.trajectory[0]
    ca_ag      = u.select_atoms(CA_SELECTION)
    ca_resids  = np.array([a.resid for a in ca_ag], dtype=int)
    ca_coords  = CA_ALIGNED[condition]
    npdf       = _extract_entities(u, condition)
    nped       = _nearest_contacts(u, npdf, ca_coords, ca_resids, NPN_CUTOFFS, NPN_TOP_K)
    NONPROTEIN_NODES[condition] = npdf
    NP_CONTACT_EDGES[condition] = nped
    npdf.to_csv(EXPORT_DIR / f'nonprotein_nodes_{condition}.csv', index=False)
    nped.to_csv(EXPORT_DIR / f'np_contact_edges_{condition}.csv', index=False)
    print(f'[NPN] {condition}: lipid={len(npdf[npdf.node_type=="lipid"])} '
          f'glycan={len(npdf[npdf.node_type=="glycan"])} '
          f'ligand={len(npdf[npdf.node_type=="ligand"])} '
          f'contacts={len(nped)}')


[NPN] normal: lipid=800 glycan=84 ligand=0 contacts=298
[NPN] tumor: lipid=788 glycan=60 ligand=0 contacts=406


## Step 10. HeteroGraph (optional, USE_HETERO=True)

**v1.1:** при `USE_HETERO=True` граф получает `data.global_features = [rg_mean, rg_std]` — graph-level признак, не привязанный к конкретному узлу.


In [ ]:
if not USE_HETERO:
    print("USE_HETERO=False — HeteroGraph пропущен. "
          "Используй contacts_raw CSV для анализа.")
else:
    if not _TORCH_OK:
        raise RuntimeError("USE_HETERO=True, но PyTorch/PyG не установлены.")

    def build_heterograph(condition):
        import torch as _t
        data       = HeteroData()
        resid_list = ESM_DATA[condition]['resid_list']
        sf         = STRUCTURAL_FEATURES[condition]

        # ── Node features (protein) ───────────────────────────────────────
        node_feats = np.concatenate([
            sf['rmsf'], sf['sasa'], sf['dssp'], sf['phi_psi'], sf['tm_relative_z']
        ], axis=1)
        if ESM_DATA[condition]['embeddings'] is not None:
            node_feats = np.concatenate([node_feats, ESM_DATA[condition]['embeddings']], axis=1)
        data['protein'].x     = _t.tensor(node_feats, dtype=_t.float32)
        data['protein'].resid = _t.tensor(resid_list, dtype=_t.long)

        # ── Edges protein–protein ─────────────────────────────────────────
        for _, edge_type, _ in RELATION_TYPES:
            if edge_type in CONTACT_MAPS[condition]:
                cm = CONTACT_MAPS[condition][edge_type]
                data['protein', edge_type, 'protein'].edge_index = cm['edge_index']
                data['protein', edge_type, 'protein'].edge_attr  = cm['edge_attr']

        # ── Graph-level features: Rg (v1.1 NEW) ─────────────────────────
        rg_mean = sf.get('rg_mean', float('nan'))
        rg_std  = sf.get('rg_std', 0.0)
        data.global_features = _t.tensor([rg_mean, rg_std], dtype=_t.float32)
        # global_features[0] = Rg_mean (Å),  global_features[1] = Rg_std (Å)

        return data

    HETERO_GRAPHS = {}
    for condition in UNIVERSES:
        HETERO_GRAPHS[condition] = build_heterograph(condition)
        g = HETERO_GRAPHS[condition]
        print(f'[{condition}] protein nodes: {g["protein"].x.shape[0]} | '
              f'features: {g["protein"].x.shape[1]} | '
              f'Rg_mean: {g.global_features[0]:.3f} Å | '
              f'Rg_std: {g.global_features[1]:.3f} Å')


USE_HETERO=False — HeteroGraph пропущен. Используй contacts_raw CSV для анализа.


## Step 11. Artefact export (residue_table, contact_edges, rg_timeseries, manifest)

**v1.1:** добавлен `rg_timeseries_{condition}.csv`; секция `global_features` в `mania_manifest.json`.

**v1.2**: в список файлов манифеста добавлен `edge_semantics.json`.

In [ ]:
# ============================================================
# MANIA preprocessing — Step 11: Artefact export
# Новый формат:
#   CONTACT_MAPS[condition] = pd.DataFrame
#   CONTACT_TENSORS[condition] = dict(edge_type -> {'edge_index','edge_attr'})
# ============================================================

CONTACT_TENSORS = {}

for condition in UNIVERSES:
    resid_list = ESM_DATA[condition]['resid_list']
    resname_list = ESM_DATA[condition]['resname_list']
    sf = STRUCTURAL_FEATURES[condition]

    resid_to_idx = {int(resid): idx for idx, resid in enumerate(resid_list)}

    # ── Таблица остатков ──────────────────────────────────────────────────────
    rows = []
    dssp_labels = ['H', 'B', 'E', 'G', 'I', 'T', 'S', 'C']

    for idx, resid in enumerate(resid_list):
        xyz = CA_COORDS[condition].get(int(resid), [0.0, 0.0, 0.0])
        ss = dssp_labels[int(np.argmax(sf['dssp'][idx]))]

        rows.append({
            'resid': int(resid),
            'resname': resname_list[idx],
            'rmsf_A': round(float(sf['rmsf'][idx, 0]), 4),
            'sasa_A2': round(float(sf['sasa'][idx, 0]), 4),
            'ss': ss,
            'x_ca': round(float(xyz[0]), 3),
            'y_ca': round(float(xyz[1]), 3),
            'z_ca': round(float(xyz[2]), 3),
            'condition': condition,
        })

    df_res = pd.DataFrame(rows)
    df_res.to_csv(EXPORT_DIR / f'residue_table_{condition}.csv', index=False)

    # ── Таблица контактных рёбер из DataFrame ────────────────────────────────
    edges_df = CONTACT_MAPS[condition].copy()

    if len(edges_df) == 0:
        print(f'[edges] {condition}: CONTACT_MAPS пуст')
        df_edges = pd.DataFrame(columns=[
            'i', 'j', 'resid_i', 'resid_j', 'edge_type',
            'contact_freq', 'mean_dist_A', 'std_dist_A', 'condition'
        ])
        CONTACT_TENSORS[condition] = {}
    else:
        # resid -> node index для ST-GNN
        edges_df['i'] = edges_df['resid_i'].map(resid_to_idx)
        edges_df['j'] = edges_df['resid_j'].map(resid_to_idx)

        missing_mask = edges_df['i'].isna() | edges_df['j'].isna()
        if missing_mask.any():
            print(f'[edges] {condition}: WARNING {missing_mask.sum()} edges with missing resid->idx mapping dropped')
            edges_df = edges_df.loc[~missing_mask].copy()

        edges_df['i'] = edges_df['i'].astype(int)
        edges_df['j'] = edges_df['j'].astype(int)
        edges_df['condition'] = condition

        df_edges = edges_df[[
            'i', 'j', 'resid_i', 'resid_j', 'edge_type',
            'contact_freq', 'mean_dist_A', 'std_dist_A', 'condition'
        ]].copy()

        df_edges.to_csv(
            EXPORT_DIR / f'protein_contact_edges_undirected_{condition}.csv',
            index=False
        )

        print(f'[edges] {condition}: {len(df_edges)} edges exported')
        print(f'         edge types: {df_edges["edge_type"].value_counts().to_dict()}')

        # ── CONTACT_TENSORS для ST-GNN ───────────────────────────────────────
        CONTACT_TENSORS[condition] = {}

        for edge_type in sorted(df_edges['edge_type'].unique()):
            sub = df_edges[df_edges['edge_type'] == edge_type].copy()

            # undirected graph -> добавляем обратные рёбра для PyG
            edge_pairs = sub[['i', 'j']].to_numpy(dtype=np.int64)
            edge_pairs_rev = edge_pairs[:, ::-1]
            edge_index_np = np.vstack([edge_pairs, edge_pairs_rev]).T  # shape (2, 2E)

            edge_attr_np = sub[['contact_freq', 'mean_dist_A', 'std_dist_A']].to_numpy(dtype=np.float32)
            edge_attr_np = np.vstack([edge_attr_np, edge_attr_np])  # дублируем для обратных рёбер

            CONTACT_TENSORS[condition][edge_type] = {
                'edge_index': edge_index_np,
                'edge_attr': edge_attr_np,
            }

            print(
                f'         tensor[{edge_type}]: '
                f'edge_index {edge_index_np.shape}, edge_attr {edge_attr_np.shape}'
            )

    if len(edges_df) == 0:
        df_edges.to_csv(
            EXPORT_DIR / f'protein_contact_edges_undirected_{condition}.csv',
            index=False
        )

    # ── Rg timeseries (v1.1 NEW) ─────────────────────────────────────────────
    if COMPUTE_RG and len(sf.get('rg_timeseries', [])) > 0:
        df_rg = pd.DataFrame({
            'frame': sf['rg_frames'],
            'rg_A': sf['rg_timeseries'],
            'condition': condition,
        })
        df_rg.to_csv(EXPORT_DIR / f'rg_timeseries_{condition}.csv', index=False)
        print(
            f'[Rg] {condition}: {len(df_rg)} frames → rg_timeseries_{condition}.csv '
            f'(mean={sf["rg_mean"]:.3f} Å, std={sf["rg_std"]:.3f} Å)'
        )

# ── Manifest + config ───────────────────────────────────────────────────────
global_features_report = {}
for condition in UNIVERSES:
    sf = STRUCTURAL_FEATURES[condition]
    global_features_report[condition] = {
        'rg_mean_A': round(sf.get('rg_mean', float('nan')), 4),
        'rg_std_A': round(sf.get('rg_std', 0.0), 4),
        'n_rg_frames': int(len(sf.get('rg_timeseries', []))),
    }

manifest = {
    'version': '1.1',
    'conditions': list(UNIVERSES.keys()),
    'global_features': global_features_report,
    'residue_qc': {
        'library_json': str(RESIDUE_LIB_PATH),
        'validated': RESIDUE_LIB_VALIDATE,
    },
    'files': [
        p.name for p in sorted(EXPORT_DIR.iterdir())
        if p.suffix in {'.csv', '.json', '.npy', '.parquet'}
    ],
}
with open(EXPORT_DIR / 'mania_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

config_snapshot = {
    'version': '1.1',
    'feature_calc_params': FEATURE_CALC_PARAMS,
    'compute_rg': COMPUTE_RG,
    'residue_lib_validate': RESIDUE_LIB_VALIDATE,
    'residue_library_json': str(RESIDUE_LIB_PATH),
    'conditions': {
        cond: {'label': CONFIG[cond]['label'], 'ph': CONFIG[cond]['ph']}
        for cond in CONFIG
    },
    'protein_selection': PROTEIN_SELECTION,
    'lipid_selection': LIPID_SELECTION,
    'glycan_resnames': list(GLYCAN_RESNAMES),
    'regions_of_interest': REGIONS_OF_INTEREST,
    'use_esm2': USE_ESM2,
    'use_hetero': USE_HETERO,
    'backbone_max_ca_dist_A': BACKBONE_MAX_CA_DIST_A,
}
with open(EXPORT_DIR / 'mania_config.json', 'w') as f:
    json.dump(config_snapshot, f, indent=2, ensure_ascii=False)

print("\nЭкспортированные артефакты:")
for p in sorted(EXPORT_DIR.iterdir()):
    print(f"  {p.name}  {p.stat().st_size / 1024:.1f} KB")

[edges] normal: 5803 edges exported
         edge types: {'vdw': 3263, 'hbond': 1693, 'backbone': 689, 'hydrophobic': 84, 'aromatic_pi': 24, 'ionic': 22, 'salt_bridge': 21, 'cation_pi': 7}
         tensor[aromatic_pi]: edge_index (2, 48), edge_attr (48, 3)
         tensor[backbone]: edge_index (2, 1378), edge_attr (1378, 3)
         tensor[cation_pi]: edge_index (2, 14), edge_attr (14, 3)
         tensor[hbond]: edge_index (2, 3386), edge_attr (3386, 3)
         tensor[hydrophobic]: edge_index (2, 168), edge_attr (168, 3)
         tensor[ionic]: edge_index (2, 44), edge_attr (44, 3)
         tensor[salt_bridge]: edge_index (2, 42), edge_attr (42, 3)
         tensor[vdw]: edge_index (2, 6526), edge_attr (6526, 3)
[Rg] normal: 301 frames → rg_timeseries_normal.csv (mean=36.438 Å, std=1.269 Å)
[edges] tumor: 5914 edges exported
         edge types: {'vdw': 3330, 'hbond': 1716, 'backbone': 689, 'hydrophobic': 99, 'ionic': 26, 'aromatic_pi': 26, 'salt_bridge': 22, 'cation_pi': 6}
         t

## Step 12. Upload to Yandex.Disk

In [ ]:
import yadisk as _yadisk_upload

client = _yadisk_upload.Client(token=MW_TOKEN)

remote_dir = ("disk:/MANIA_WANIA_project/MD_trajectories/NaPi2b/Ramilia/"
              "30ns_03-2026/results/preprocessing_export_v1.2")

if not client.exists(remote_dir):
    client.makedirs(remote_dir)
    print(f"Папка создана: {remote_dir}")

for fpath in sorted(EXPORT_DIR.iterdir()):
    if fpath.suffix in {'.csv', '.json', '.npy', '.parquet'}:
        remote_path = f'{remote_dir}/{fpath.name}'
        print(f'  ↑ {fpath.name} ...')
        client.upload(str(fpath), remote_path, overwrite=True)
        print(f'  [ok] {fpath.name}')

print("Загрузка завершена.")

Папка создана: disk:/MANIA_WANIA_project/MD_trajectories/NaPi2b/Ramilia/30ns_03-2026/results/preprocessing_export_v1.2
  ↑ contacts_perframe_normal.parquet ...
  [ok] contacts_perframe_normal.parquet
  ↑ contacts_perframe_tumor.parquet ...
  [ok] contacts_perframe_tumor.parquet
  ↑ edge_semantics.json ...
  [ok] edge_semantics.json
  ↑ mania_config.json ...
  [ok] mania_config.json
  ↑ mania_manifest.json ...
  [ok] mania_manifest.json
  ↑ mania_residue_library.json ...
  [ok] mania_residue_library.json
  ↑ nonprotein_nodes_normal.csv ...
  [ok] nonprotein_nodes_normal.csv
  ↑ nonprotein_nodes_tumor.csv ...
  [ok] nonprotein_nodes_tumor.csv
  ↑ np_contact_edges_normal.csv ...
  [ok] np_contact_edges_normal.csv
  ↑ np_contact_edges_tumor.csv ...
  [ok] np_contact_edges_tumor.csv
  ↑ protein_contact_edges_undirected_normal.csv ...
  [ok] protein_contact_edges_undirected_normal.csv
  ↑ protein_contact_edges_undirected_tumor.csv ...
  [ok] protein_contact_edges_undirected_tumor.csv
  ↑ res